# Article text representations

BT4222 Group 6 · representations 1/3.

Fit **title TF-IDF + SVD on training news only**, freeze the transform, then encode title and abstract
in that same space. Export title, abstract and fused text vectors, article-row mappings and title token
sequences for the NRMS owner. This is a reproducible compact baseline, not a trained NRMS model.

Run top to bottom. Shared algorithms, validation and file handling live in `src/newsrec/`; this notebook controls settings and shows results. You do not run those Python files separately. Saved artifacts and `REBUILD` work as before.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents,
                            Path('/content/drive/MyDrive/BT4222'))
                  if (p / 'src/newsrec').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project checkout containing src/newsrec/.')
sys.path.insert(0, str(CODE_ROOT / 'src'))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = None  # Optional: point to another folder holding the project's saved datasets.
ROOT = Path(DATA_ROOT or os.environ.get('NEWSREC_ROOT') or CODE_ROOT).expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Data/output root:', ROOT)

from newsrec.representations import build_news, representation_summary, text_neighbors

In [ ]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Fit and export

Defaults: title unigrams/bigrams, up to 50,000 terms, `min_df=2`, SVD64, fixed seed.
All vectors are float32 and L2-normalised when valid. The combined view weights title/abstract 0.7/0.3;
a missing/OOV field falls back to the other valid field. Completely unsupported text remains zero with a mask.
Array row 0 is reserved for padding; real `news_code` values must go through `article_index.parquet`.
Training-fitted category/subcategory vocabularies reserve code 0 for unseen values.

Rebuilding representations invalidates dependent artifacts. Use one settings across the three notebooks.

In [ ]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "news"):
    coverage = build_news(ROOT, SETTINGS)
else:
    print("Loaded saved article text representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("text")]
display(coverage)

## 2. Check dimensions, norms and quality

A present title can become a zero vector after train-vocabulary filtering. Abstract absence and zero-vector
rates are distinct. On a tiny fixture the effective SVD rank can be smaller than the configured dimension;
unused dimensions are padded with zeros and the effective rank is recorded.

In [ ]:
display(representation_summary(ROOT, SETTINGS))
index = pd.read_parquet(ARTIFACTS / 'article_index.parquet')
display(index[['news_code', 'title', 'has_abstract', 'has_title_vector', 'has_abstract_vector']].head())
meta = json.loads((ARTIFACTS / 'news_manifest.json').read_text())
display({k: meta[k] for k in ['effective_svd_dim', 'fit_scope', 'token_oov_fraction', 'title_truncation_fraction']})

## 3. Inspect neighbours without using click labels

This is a qualitative sanity check of shared text coordinates, not evidence of a ranking gain.
The frozen vocabulary can leave new articles unsupported; the serving pipeline retains those articles
and uses entity/popularity/freshness fallback rather than dropping them.

In [ ]:
display(text_neighbors(ROOT, count=5))
display(pd.read_csv(ROOT / 'results/representation_cost.csv').tail())

## Handoff

Consumers can load the saved arrays with NumPy and the article index with pandas after all three notebooks finish.
`title64` reproduces the report's multi-interest space; `text64` is a separate combined-content view.
NRMS receives `title_token_ids.npy`, its mask, and the training-only vocabulary (`0=PAD`, `1=UNK`);
its attention encoders and learned embeddings belong to the NRMS notebook.